# Mission 1 — Exploration et préparation des données

**Projet ImmoPrix** — Prédiction du prix médian des maisons en Californie.

Cible : `MedHouseVal` (prix médian en 100K$).

Features :
- `MedInc` — Revenu médian (10K$)
- `HouseAge` — Âge médian des maisons
- `AveRooms` — Pièces par logement
- `AveBedrms` — Chambres par logement
- `Population` — Population du secteur
- `AveOccup` — Occupation moyenne
- `Latitude`, `Longitude` — Coordonnées géographiques

## 1. Chargement des données

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

In [2]:
# Chargement du dataset California Housing depuis scikit-learn
housing = fetch_california_housing(as_frame=True)

df = housing.frame
df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


## 2. Persistance en base (DuckDB)

On stocke le dataset dans une base DuckDB locale (`data/raw/california_housing.db`) pour que les notebooks suivants (exploration, modélisation, monitoring) ne refassent pas l'appel à `fetch_california_housing` et travaillent tous sur la même source.

Table cible : `housing` (features + cible `MedHouseVal`).

In [3]:
from pathlib import Path
import duckdb

DB_PATH = Path("../data/raw/california_housing.db")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(str(DB_PATH))
con.execute("CREATE OR REPLACE TABLE housing AS SELECT * FROM df")
con.close()

print(f"Écrit dans {DB_PATH.resolve()}")

Écrit dans /home/d/l/dlahanqu/Documents/IS2A/MLOPS/data/raw/california_housing.db


In [4]:
# Vérification : on relit depuis DuckDB
con = duckdb.connect(str(DB_PATH), read_only=True)
n_rows = con.sql("SELECT COUNT(*) AS n FROM housing").fetchone()[0]
df_check = con.sql("SELECT * FROM housing LIMIT 5").df()
con.close()

print(f"Lignes en base : {n_rows}")
df_check

Lignes en base : 20640


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


### Comment relire ces données depuis un autre notebook

Dans `notebooks/exploration.ipynb`, `notebooks/modelisation.ipynb`, etc. :

```python
import duckdb
from pathlib import Path

DB_PATH = Path("../data/raw/california_housing.db")

con = duckdb.connect(str(DB_PATH), read_only=True)
df = con.sql("SELECT * FROM housing").df()
con.close()
```

Plus besoin de rappeler `fetch_california_housing`.